In [ ]:
!pip install codecarbon -q

In [ ]:
from codecarbon import EmissionsTracker
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split
import time
import torch
from torch.amp import autocast, GradScaler
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

import sys
sys.path.append('.')
from config import PROJECT_PATH

## Configuration

This notebook trains AlexNet and DenseNet-121 on chest X-ray radiographs from the VinDr-CXR dataset at two resolutions (256×256 and 1024×1024) for multi-label classification of aortic enlargement and cardiomegaly.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
!unzip -q {PROJECT_PATH}/dataset.zip -d {PROJECT_PATH}

In [ ]:
df = pd.read_csv(f'{PROJECT_PATH}/dataset/metadata_1024.csv')

In [ ]:
# Classify each image into a category for the split
def category_split(image_id, ids_aneurysm, ids_cardiomegaly):
    in_aneurysm = image_id in ids_aneurysm
    in_cardio = image_id in ids_cardiomegaly
    if in_aneurysm and in_cardio:
        return 'both'
    elif in_aneurysm:
        return 'aortic enlargement'
    elif in_cardio:
        return 'cardiomegaly'
    else:
        return 'healthy'

# Create DataFrame of unique images with their category 
ids_aneurysm = set(df[df['class_id'] == 0]['image_id'])
ids_cardiomegaly = set(df[df['class_id'] == 1]['image_id'])

df_unique = pd.DataFrame({'image_id': df['image_id'].unique()})
df_unique['category'] = df_unique['image_id'].apply(
    lambda x: category_split(x, ids_aneurysm, ids_cardiomegaly)
)

print("Category Distribution:")
print(df_unique['category'].value_counts())
print(f"Total unique images: {len(df_unique)}")

# Stratified split 80/10/10
train_ids, temp_ids = train_test_split(
    df_unique, test_size=0.2, stratify=df_unique['category'], random_state=42
)
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.5, stratify=temp_ids['category'], random_state=42
)

print(f"\nTrain: {len(train_ids)} images")
print(f"Val:   {len(val_ids)} images")
print(f"Test:  {len(test_ids)} images")

# Save splits as a fixed csv
train_ids[['image_id']].to_csv(f'{PROJECT_PATH}/dataset/split_train.csv', index=False)
val_ids[['image_id']].to_csv(f'{PROJECT_PATH}/dataset/split_val.csv', index=False)
test_ids[['image_id']].to_csv(f'{PROJECT_PATH}/dataset/split_test.csv', index=False)

print("\nSplits saved")

In [ ]:
# N-CLAHE function
def apply_nclahe(image_np, tile_size):
    image_log = np.log1p(image_np.astype(np.float32))
    image_log = ((image_log - image_log.min()) /
                  (image_log.max() - image_log.min()) * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(tile_size, tile_size))
    return clahe.apply(image_log)

## Reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

## Dataset Class

In [ ]:
#Dataset class
class RadiographyDataset(Dataset):
    def __init__(self, metadata_csv, split_csv, images_dir, resolution, is_train=False):
        self.df = pd.read_csv(metadata_csv)
        self.split_ids = pd.read_csv(split_csv)['image_id'].tolist()
        self.df = self.df[self.df['image_id'].isin(self.split_ids)]
        self.images_dir = images_dir
        self.resolution = resolution
        self.is_train = is_train
        self.tile_size = 4 if resolution == 256 else 16

        self.image_ids = self.df['image_id'].unique().tolist()

        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        self.augment = transforms.Compose([
            transforms.RandomRotation(degrees=5),
            transforms.RandomResizedCrop(resolution, scale=(0.9, 1.0)),
        ]) if is_train else None

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]

        img = cv2.imread(f'{self.images_dir}/{image_id}.png', cv2.IMREAD_GRAYSCALE)
        img = apply_nclahe(img, self.tile_size)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = Image.fromarray(img)

        if self.is_train and self.augment:
            img = self.augment(img)

        img = transforms.ToTensor()(img)
        img = self.normalize(img)

        # Multi-label tags [aneurysm, cardiomegaly]
        filas = self.df[self.df['image_id'] == image_id]
        label = torch.zeros(2)
        if 0 in filas['class_id'].values:
            label[0] = 1.0
        if 1 in filas['class_id'].values:
            label[1] = 1.0

        return img, label


# Dataloaders function
def create_dataloaders(metadata_csv, images_dir, resolution, batch_size):

    train_dataset = RadiographyDataset(
        metadata_csv, f'{PROJECT_PATH}/split_train.csv', images_dir, resolution, is_train=True
    )
    val_dataset = RadiographyDataset(
        metadata_csv, f'{PROJECT_PATH}/split_val.csv', images_dir, resolution, is_train=False
    )
    test_dataset = RadiographyDataset(
        metadata_csv, f'{PROJECT_PATH}/split_test.csv', images_dir, resolution, is_train=False
    )

    # WeightedRandomSampler to balance training set
    labels = []
    for image_id in train_dataset.image_ids:
        filas = train_dataset.df[train_dataset.df['image_id'] == image_id]
        es_clinica = (0 in filas['class_id'].values) or (1 in filas['class_id'].values)
        labels.append(1 if es_clinica else 0)

    n_clinical = sum(labels)
    n_healthy = len(labels) - n_clinical
    class_weights = [1.0 / n_healthy, 1.0 / n_clinical]
    sample_weights = [class_weights[l] for l in labels]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(train_dataset),
        replacement=True
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"Train: {len(train_dataset)} imágenes")
    print(f"Val:   {len(val_dataset)} imágenes")
    print(f"Test:  {len(test_dataset)} imágenes")

    return train_loader, val_loader, test_loader

In [ ]:
def build_model(architecture, resolution, freeze_backbone=True):
    """
    architecture: 'alexnet' o 'densenet'
    resolution: 256 o 1024
    freeze_backbone: si True congela todo excepto la última capa
    """
    if architecture == 'alexnet':
        model = models.alexnet(weights='IMAGENET1K_V1')
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        # Substitute final classifier
        model.classifier[6] = nn.Linear(4096, 2)

        # For higher resolution alexnet unfreeze last three feature layers
        if resolution == 1024:
            for name, param in model.named_parameters():
                if any(f'features.{i}' in name for i in [10, 11, 12]):
                    param.requires_grad = True

    elif architecture == 'densenet':
        model = models.densenet121(weights='IMAGENET1K_V1')
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        # Substitute final classifier
        model.classifier = nn.Linear(1024, 2)

        # For higher resolution densenet-121 unfreeze last dense block and norm layer
        if resolution == 1024:
            for name, param in model.named_parameters():
                if 'denseblock4' in name or 'norm5' in name:
                    param.requires_grad = True

    return model.to(DEVICE)


def train_model(model, train_loader, val_loader, architecture, resolution,
                    epochs=30, patience=5):

    lr = 1e-4 if resolution == 256 else 1e-5
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    scaler = GradScaler()  # Mixed precision

    best_val_loss = float('inf')
    epochs_without_improvement = 0
    record = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        t0 = time.time()

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE
            )
            optimizer.zero_grad()

            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                with autocast():
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append((outputs.cpu().numpy() > 0.5).astype(int))
                all_labels.append(labels.cpu().numpy())

        val_loss /= len(val_loader)
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)

        mcc_aneurysm = matthews_corrcoef(all_labels[:, 0], all_preds[:, 0])
        mcc_cardio = matthews_corrcoef(all_labels[:, 1], all_preds[:, 1])

        print(f"Epoch {epoch+1}/{epochs} | "
        f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
        f"MCC [Aneu: {mcc_aneurysm:.3f} Card: {mcc_cardio:.3f}] | "
        f"Time: {time.time()-t0:.1f}s")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(),
                      f'{PROJECT_PATH}/{architecture}_{resolution}_best.pth')
            epochs_without_improvement = 0
            print(f"   ✓ Best model saved")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"   Early stopping in epoch {epoch+1}")
                break

    return record

## AlexNet 256x256


In [ ]:
# Create dataloaders
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{PROJECT_PATH}/dataset/metadata_256.csv',
    images_dir=f'{PROJECT_PATH}/dataset/images_256',
    resolution=256,
    batch_size=16
)

In [ ]:
# Build AlexNet
model = build_model('alexnet', resolution=256)

# Verify trainable parameters and freezed parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

# Train and monitor AlexNet with 256x256 images
tracker = EmissionsTracker()
tracker.start()

record = train_model(model, train_loader, val_loader,
                        architecture='alexnet', resolution=256)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## DenseNet-121 256x256

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{PROJECT_PATH}/dataset/metadata_256.csv',
    images_dir=f'{PROJECT_PATH}/dataset/images_256',
    resolution=256,
    batch_size=16
)

In [ ]:

model = build_model('densenet', resolution=256)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

tracker = EmissionsTracker()
tracker.start()

record = train_model(model, train_loader, val_loader,
                        architecture='densenet', resolution=256, epochs=50)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## AlexNet 1024x1024


In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{PROJECT_PATH}/dataset/metadata_1024.csv',
    images_dir=f'{PROJECT_PATH}/dataset/images_1024',
    resolution=1024,
    batch_size=4
)

In [ ]:
model = build_model('alexnet', resolution=1024)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

tracker = EmissionsTracker()
tracker.start()

record = train_model(model, train_loader, val_loader,
                        architecture='alexnet', resolution=1024,
                            epochs=50, patience=5)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")

## DenseNet-121 1024x1024

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    metadata_csv=f'{PROJECT_PATH}/dataset/metadata_1024.csv',
    images_dir=f'{PROJECT_PATH}/dataset/images_1024',
    resolution=1024,
    batch_size=4
)

In [ ]:
model = build_model('densenet', resolution=1024)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

tracker = EmissionsTracker(log_level="error")
tracker.start()

record = train_model(model, train_loader, val_loader,
                        architecture='densenet', resolution=1024,
                            epochs=50, patience=5)

emissions = tracker.stop()
print(f"CO2 emissions: {emissions:.6f} kg")